In [2]:
#!pip install pandas datasets scikit-learn

## Descriere dataset

Dataset-ul utilizat contine CV-uri provenite din surse publice disponibile online (HuggingFace dataset).

### Caracteristici:
- aproximativ 500 CV-uri utilizate pentru analiza
- datele sunt in format text (nestructurat)

### Coloane:
- resume_text – textul complet al CV-ului
- clean_resume – versiunea curatata a textului
- skills_found – lista de skill-uri extrase
- num_skills – numarul de skill-uri identificate

### Preprocesare aplicata:
- transformare text in lowercase
- eliminare caractere speciale
- eliminare spatii multiple
- extragere skill-uri prin keyword matching

### Limitari:
- unele CV-uri nu contin skill-uri detectabile
- lipsesc informatii personale (anonimizare)
- metoda de extragere este simpla (bazata pe cuvinte cheie)
- job descriptions sunt putine (doar cateva exemple demo)

In etapa urmatoare se va extinde dataset-ul si se vor folosi metode NLP mai avansate.

In [3]:
# import re
# import pandas as pd
# from datasets import load_dataset
# from collections import Counter
#
# # 1. Incarcam CV-uri de pe HuggingFace
# resumes_dataset = load_dataset("datasetmaster/resumes", split="train")
#
# # transformam in DataFrame
# resumes_df = resumes_dataset.to_pandas()
#
# print(resumes_df.head())
# print(resumes_df.columns)

# pip install pandas datasets scikit-learn

import os
import re
import pandas as pd
from datasets import load_dataset
from collections import Counter

os.makedirs("data", exist_ok=True)

MAX_RESUMES = 10000

# 1. Încărcăm CV-uri
resumes_dataset = load_dataset("datasetmaster/resumes", split="train")
resumes_df = resumes_dataset.to_pandas()

possible_resume_columns = ["resume", "text", "content", "Resume", "resume_text"]
resume_col = None

for col in possible_resume_columns:
    if col in resumes_df.columns:
        resume_col = col
        break

if resume_col is None:
    resume_col = resumes_df.columns[0]

resumes_df = resumes_df[[resume_col]].rename(columns={resume_col: "resume_text"})
resumes_df = resumes_df.dropna()
resumes_df = resumes_df.head(MAX_RESUMES)

print("Număr CV-uri folosite:", len(resumes_df))

Număr CV-uri folosite: 4817


## Incarcarea datelor

In aceasta sectiune incarcam un dataset cu CV-uri folosind biblioteca HuggingFace Datasets. Datele sunt transformate intr-un DataFrame pentru a putea fi analizate si procesate mai usor.

In [4]:
# 2. Alegem coloana cu textul CV-ului
# Daca numele coloanei difera, afisarea de mai sus va arata ce trebuie schimbat

possible_resume_columns = ["resume", "text", "content", "Resume", "resume_text"]

resume_col = None
for col in possible_resume_columns:
    if col in resumes_df.columns:
        resume_col = col
        break

if resume_col is None:
    resume_col = resumes_df.columns[0]

print("Coloana folosita pentru CV:", resume_col)

resumes_df = resumes_df[[resume_col]].rename(columns={resume_col: "resume_text"})
resumes_df = resumes_df.dropna()
resumes_df = resumes_df.head(1000)

resumes_df.head()

Coloana folosita pentru CV: resume_text


,resume_text
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
2,"{""name"":""Not Provided"",""email"":""Not Provided"",..."
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""..."


## Selectarea coloanei relevante

Dataset-ul poate avea mai multe coloane, dar pentru proiect ne intereseaza coloana care contine textul CV-ului. Aceasta este redenumita in `resume_text` pentru a fi mai usor de folosit in pasii urmatori.

In [5]:
# # 3. Curatare text
#
# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"http\S+", " ", text)
#     text = re.sub(r"[^a-zA-Z0-9+#. ]", " ", text)
#     text = re.sub(r"\s+", " ", text).strip()
#     return text
#
# resumes_df["clean_resume"] = resumes_df["resume_text"].apply(clean_text)
#
# resumes_df.head()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " email_removed ", text)
    text = re.sub(r"\+?\d[\d\s().-]{7,}", " phone_removed ", text)
    text = re.sub(r"[^a-zA-Z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


skills_list = [
    "python", "java", "c++", "c#", "sql", "mysql", "postgresql",
    "excel", "power bi", "tableau", "machine learning", "deep learning",
    "nlp", "tensorflow", "pytorch", "scikit-learn", "html", "css",
    "javascript", "typescript", "react", "node.js", "spring",
    "git", "github", "docker", "kubernetes", "linux", "aws", "azure",
    "communication", "leadership", "teamwork", "problem solving",
    "data analysis", "statistics", "streamlit", "fastapi", "flask"
]


def extract_skills(text):
    found = []
    text = str(text).lower()

    for skill in skills_list:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            found.append(skill)

    return sorted(list(set(found)))


resumes_df["clean_resume"] = resumes_df["resume_text"].apply(clean_text)
resumes_df["skills_found"] = resumes_df["clean_resume"].apply(extract_skills)
resumes_df["num_skills"] = resumes_df["skills_found"].apply(len)

resumes_df.head()

,resume_text,clean_resume,skills_found,num_skills
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...,"[c++, github, mysql, python, tensorflow]",5
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...,[],0
2,"{""name"":""Not Provided"",""email"":""Not Provided"",...",name not provided email not provided phone not...,[github],1
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...,[],0
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""...",name email phone location city pune country in...,[github],1


## Curatarea textului

Textele din CV-uri sunt transformate in litere mici si sunt eliminate caracterele speciale inutile. Acest pas ajuta la standardizarea datelor si face mai usoara identificarea skill-urilor.

In [6]:
# # 4. Lista de skill-uri cautate
#
# skills_list = [
#     "python", "java", "c++", "c#", "sql", "mysql", "postgresql",
#     "excel", "power bi", "tableau",
#     "machine learning", "deep learning", "nlp",
#     "tensorflow", "pytorch", "scikit-learn",
#     "html", "css", "javascript", "react",
#     "git", "github", "docker", "linux",
#     "communication", "leadership", "teamwork"
# ]

In [7]:
# # 5. Extragere skill-uri din fiecare CV
#
# def extract_skills(text, skills):
#     found = []
#     for skill in skills:
#         if skill in text:
#             found.append(skill)
#     return found
#
# resumes_df["skills_found"] = resumes_df["clean_resume"].apply(
#     lambda x: extract_skills(x, skills_list)
# )
#
# resumes_df["num_skills"] = resumes_df["skills_found"].apply(len)
#
# resumes_df[["clean_resume", "skills_found", "num_skills"]].head()

## Extragerea skill-urilor

Pentru fiecare CV, cautam aparitia unor skill-uri dintr-o lista predefinita. Daca un skill este gasit in text, acesta este salvat in lista de competente identificate pentru acel CV.

In [8]:
# 6. Analiza simpla a datelor

print("Numar CV-uri analizate:", len(resumes_df))
print("Lungime medie CV:", resumes_df["clean_resume"].apply(len).mean())
print("Numar mediu skill-uri gasite:", resumes_df["num_skills"].mean())

Numar CV-uri analizate: 1000
Lungime medie CV: 262.743
Numar mediu skill-uri gasite: 2.213


## Analiza datelor

In aceasta sectiune calculam statistici simple despre date: numarul de CV-uri analizate, lungimea medie a CV-urilor si numarul mediu de skill-uri gasite.

In [9]:
# 7. Top skill-uri gasite

all_skills = []

for skills in resumes_df["skills_found"]:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)

top_skills_df = pd.DataFrame(
    skill_counts.items(),
    columns=["skill", "count"]
).sort_values(by="count", ascending=False)

top_skills_df.head(10)

,skill,count
1,github,982
3,python,234
18,react,200
7,java,139
9,spring,119
8,javascript,106
17,flask,100
19,aws,83
20,docker,83
21,kubernetes,83


In [10]:
# # 8. Job descriptions demo
#
# job_descriptions = [
#     "We are looking for a Python developer with SQL, Git and machine learning knowledge.",
#     "Data analyst role requiring Excel, Power BI, SQL and communication skills.",
#     "Frontend developer with HTML, CSS, JavaScript, React and Git experience.",
#     "Backend developer with Java, Spring, SQL and Docker experience.",
#     "Machine learning engineer with Python, TensorFlow, deep learning and NLP skills.",
#     "DevOps engineer with Docker, Kubernetes, Linux and CI/CD experience.",
#     "Software engineer with C++, algorithms, data structures and problem solving skills.",
#     "Business analyst with Excel, communication, SQL and reporting experience"
# ]
#
# jobs_df = pd.DataFrame({"job_description": job_descriptions})
# jobs_df["clean_job"] = jobs_df["job_description"].apply(clean_text)
# jobs_df["required_skills"] = jobs_df["clean_job"].apply(
#     lambda x: extract_skills(x, skills_list)
# )
#
# jobs_df

job_descriptions = [
    "We are looking for a Python developer with SQL, Git and machine learning knowledge.",
    "Data analyst role requiring Excel, Power BI, SQL and communication skills.",
    "Frontend developer with HTML, CSS, JavaScript, React and Git experience.",
    "Backend developer with Java, Spring, SQL and Docker experience.",
    "Machine learning engineer with Python, TensorFlow, deep learning and NLP skills.",
    "DevOps engineer with Docker, Kubernetes, Linux and CI/CD experience.",
    "Software engineer with C++, algorithms, data structures and problem solving skills.",
    "Business analyst with Excel, communication, SQL and reporting experience"
]

jobs_df = pd.DataFrame({"job_description": job_descriptions})
jobs_df["clean_job"] = jobs_df["job_description"].apply(clean_text)
jobs_df["required_skills"] = jobs_df["clean_job"].apply(extract_skills)

jobs_df.head()

,job_description,clean_job,required_skills
0,We are looking for a Python developer with SQL...,we are looking for a python developer with sql...,"[git, machine learning, python, sql]"
1,"Data analyst role requiring Excel, Power BI, S...",data analyst role requiring excel power bi sql...,"[communication, excel, power bi, sql]"
2,"Frontend developer with HTML, CSS, JavaScript,...",frontend developer with html css javascript re...,"[css, git, html, javascript, react]"
3,"Backend developer with Java, Spring, SQL and D...",backend developer with java spring sql and doc...,"[docker, java, spring, sql]"
4,"Machine learning engineer with Python, TensorF...",machine learning engineer with python tensorfl...,"[deep learning, machine learning, nlp, python,..."


## Job Descriptions

In aceasta etapa au fost utilizate mai multe descrieri de job (exemple sintetice) pentru a testa functionalitatea sistemului.

Acestea acopera diferite roluri:
- software developer
- data analyst
- machine learning engineer
- devops engineer

Limitari:
- job descriptions sunt generate manual (nu dataset real complet)
- nu acopera toate domeniile

In etapa urmatoare se va folosi un dataset mai complex si real pentru matching.

In [11]:
# # 9. Salvam datele procesate
#
# resumes_df.to_csv("processed_resumes.csv", index=False)
# jobs_df.to_csv("processed_jobs.csv", index=False)
#
# print("Fisiere salvate: processed_resumes.csv si processed_jobs.csv")

resumes_df.to_csv("data/processed_resumes.csv", index=False)
jobs_df.to_csv("data/processed_jobs.csv", index=False)

print("Salvat: data/processed_resumes.csv")
print("Salvat: data/processed_jobs.csv")

Salvat: data/processed_resumes.csv
Salvat: data/processed_jobs.csv


## Observatii

- Unele CV-uri nu contin skill-uri detectabile din lista folosita.
- Unele informatii personale lipsesc sau sunt anonimizate.
- Datele sunt nestructurate si pot avea formate diferite.
- Pentru o extragere mai buna a informatiilor, in etapele urmatoare pot fi folosite metode NLP mai avansate.

In [12]:
# # functie pentru calcul scor potrivire
#
# def compute_match_score(cv_skills, job_skills):
#     if len(job_skills) == 0:
#         return 0
#
#     matched = set(cv_skills) & set(job_skills)
#     score = len(matched) / len(job_skills) * 100
#
#     return round(score, 2), list(matched)
def compute_match_score(cv_skills, job_skills):
    if len(job_skills) == 0:
        return 0, []

    matched = list(set(cv_skills) & set(job_skills))
    score = len(matched) / len(job_skills) * 100

    return round(score, 2), matched


matching_results = []

for job_index, job_row in jobs_df.iterrows():
    job_skills = job_row["required_skills"]

    for cv_index, cv_row in resumes_df.iterrows():
        cv_skills = cv_row["skills_found"]
        score, matched = compute_match_score(cv_skills, job_skills)
        missing = list(set(job_skills) - set(cv_skills))

        matching_results.append({
            "job_id": job_index,
            "candidate_id": cv_index,
            "job_description": job_row["job_description"],
            "match_score": score,
            "matched_skills": matched,
            "missing_skills": missing
        })

matching_df = pd.DataFrame(matching_results)
matching_df.to_csv("data/matching_results.csv", index=False)

print("Salvat: data/matching_results.csv")
print("Total matching rows:", len(matching_df))

Salvat: data/matching_results.csv
Total matching rows: 8000


In [13]:
# # luam primul CV din dataset
# cv_example = resumes_df.iloc[0]
#
# cv_skills = cv_example["skills_found"]
#
# # exemplu job
# job_skills = ["python", "sql", "docker", "git"]
#
# score, matched = compute_match_score(cv_skills, job_skills)
#
# missing = list(set(job_skills) - set(cv_skills))
#
# print("Candidate Match Score:", score, "%")
# print("Matched Skills:", matched)
# print("Missing Skills:", missing)

training_pairs = []

for job_index, job_row in jobs_df.iterrows():
    job_skills = job_row["required_skills"]

    for cv_index, cv_row in resumes_df.iterrows():
        cv_skills = cv_row["skills_found"]
        score, matched = compute_match_score(cv_skills, job_skills)

        label = score / 100

        training_pairs.append({
            "job_id": job_index,
            "candidate_id": cv_index,
            "cv_text": cv_row["clean_resume"],
            "job_text": job_row["clean_job"],
            "cv_skills": cv_skills,
            "job_skills": job_skills,
            "label": round(label, 4)
        })

training_pairs_df = pd.DataFrame(training_pairs)

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    training_pairs_df,
    test_size=0.3,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

final_training_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
final_training_df.to_csv("data/training_pairs.csv", index=False)

print("Salvat: data/training_pairs.csv")
print(final_training_df["split"].value_counts())

Salvat: data/training_pairs.csv
split
train    5600
val      1200
test     1200
Name: count, dtype: int64


## Exemplu output sistem

Pentru un candidat analizat:

- Candidate Match Score: procentul de skill-uri din job regasite in CV
- Matched Skills: skill-uri comune intre CV si job
- Missing Skills: skill-uri necesare dar absente

Acest tip de output ofera suport decizional pentru recrutori.

In [14]:
# # test pe mai multi candidati
#
# for i in range(3):
#     cv = resumes_df.iloc[i]
#     score, matched = compute_match_score(cv["skills_found"], job_skills)
#
#     print("\nCandidat", i+1)
#     print("Scor:", score, "%")
#     print("Skill-uri potrivite:", matched)

## Matching automat pentru toate job descriptions

In aceasta sectiune testam sistemul pe mai multe descrieri de job. Pentru fiecare job, calculam scorul de potrivire pentru fiecare CV si afisam cei mai potriviti candidati.

In [15]:
# matching_results = []
#
# for job_index, job_row in jobs_df.iterrows():
#     job_skills = job_row["required_skills"]
#
#     for cv_index, cv_row in resumes_df.iterrows():
#         cv_skills = cv_row["skills_found"]
#
#         score, matched = compute_match_score(cv_skills, job_skills)
#
#         # calculam missing separat
#         missing = list(set(job_skills) - set(cv_skills))
#
#         matching_results.append({
#             "job_id": job_index,
#             "candidate_id": cv_index,
#             "job_description": job_row["job_description"],
#             "match_score": score,
#             "matched_skills": matched,
#             "missing_skills": missing
#         })
#
# matching_df = pd.DataFrame(matching_results)
#
# matching_df.head()

In [16]:
# top_candidates = matching_df.sort_values(
#     by=["job_id", "match_score"],
#     ascending=[True, False]
# ).groupby("job_id").head(3)
#
# top_candidates[[
#     "job_id",
#     "candidate_id",
#     "match_score",
#     "matched_skills",
#     "missing_skills"
# ]]

In [17]:
# for job_id in jobs_df.index:
#     print("\n==============================")
#     print("JOB", job_id)
#     print(jobs_df.loc[job_id, "job_description"])
#     print("==============================")
#
#     top = top_candidates[top_candidates["job_id"] == job_id]
#
#     for _, row in top.iterrows():
#         print("\nCandidat:", row["candidate_id"])
#         print("Candidate Match Score:", row["match_score"], "%")
#         print("Matched Skills:", row["matched_skills"])
#         print("Missing Skills:", row["missing_skills"])

## Matching CV - Job Description

Pentru fiecare CV si fiecare job description se calculeaza un scor de potrivire.

Scorul reprezinta procentul de skill-uri cerute de job care sunt gasite in CV.

- Matched Skills: skill-uri comune intre CV si job
- Missing Skills: skill-uri necesare dar absente din CV

Aceasta metoda este una simpla, bazata pe keyword matching, si va fi imbunatatita in etapele urmatoare folosind tehnici NLP mai avansate.

In [18]:
# def generate_matching_results(resumes_df, jobs_df, output_file="matching_results.csv"):
#
#     matching_results = []
#
#     for job_index, job_row in jobs_df.iterrows():
#         job_skills = job_row["required_skills"]
#
#         for cv_index, cv_row in resumes_df.iterrows():
#             cv_skills = cv_row["skills_found"]
#
#             score, matched = compute_match_score(cv_skills, job_skills)
#
#             # calculam missing skills
#             missing = list(set(job_skills) - set(cv_skills))
#
#             matching_results.append({
#                 "job_id": job_index,
#                 "candidate_id": cv_index,
#                 "job_description": job_row["job_description"],
#                 "match_score": score,
#                 "matched_skills": matched,
#                 "missing_skills": missing
#             })
#
#     matching_df = pd.DataFrame(matching_results)
#
#     # salvare CSV
#     matching_df.to_csv(output_file, index=False)
#
#     print(f"Fisier salvat: {output_file}")
#
#     return matching_df

In [19]:
# matching_df = generate_matching_results(resumes_df, jobs_df)